# Part 1 · Data understanding

Audit of `sample_data.csv`: what a row is, what the data covers, what every column is and when it is known, the as-of policy and leakage audit, and the data issues that would change a business decision.

All logic lives in `src/sciplay/` (`audit.py`, `registry.py`, `asof.py`); this notebook only calls it. Nothing is cleaned here: it describes the raw file.

**Contents**
1. Source grain and duplicate keys
2. Coverage: dates, players, cohorts
3. Field groups, missing values and inconsistent semantics
4. Column registry: identifiers, dimensions, measures, derived features, targets, excluded
5. As-of policy, with examples
6. Leakage audit, with examples
7. Late data, multiple platforms, sparse fields
8. Data quality issues
9. Definitions
10. Summary

In [1]:
import pandas as pd
import plotly.io as pio
from IPython.display import display

from sciplay.audit import checks as audit
from sciplay.data import asof, registry
from sciplay.viz import charts as pc
from sciplay.config import REPO_ROOT, get_settings
from sciplay.data.loader import load_raw

pio.renderers.default = 'plotly_mimetype+notebook_connected'
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 200)

df = load_raw(get_settings().raw_data_path)
profile = audit.column_profile(df)
reg = registry.build_registry(df, profile)
players = audit.player_summary(df)
audit.dataset_overview(df)

,metric,value
0,Rows,4245
1,Columns,409
2,Distinct players,700
3,Distinct data dates,180
4,First data date,2026-05-09
5,Last data date,2026-11-04
6,First install date,2026-05-09
7,Last install date,2026-11-04
8,Max days since install,169
9,Platforms,6


## 1. Source grain and duplicate keys

A key is valid when no two rows share it.

In [2]:
audit.duplicate_keys(df)

,candidate_key,distinct_keys,rows_in_duplicated_keys,extra_rows,is_unique
0,DATE + PLAYER_ID + PLATFORM_ID,4245,0,0,True
1,DATE + PLAYER_ID,4238,14,7,False
2,PLAYER_ID + PLATFORM_ID,703,3910,3542,False
3,PLAYER_ID,700,3913,3545,False
4,Full row (all columns),4245,0,0,True


In [3]:
ev = audit.grain_evidence(df)
pd.DataFrame({
    'check': ['Rows with SESSIONS > 0', 'DAYS_SINCE_LAST_LOGIN equals the gap to the previous row'],
    'share': [ev.rows_with_sessions_share, ev.login_gap_matches_row_gap_share],
}).style.format({'share': '{:.1%}'})

,check,share
0,Rows with SESSIONS > 0,99.9%
1,DAYS_SINCE_LAST_LOGIN equals the gap to the previous row,99.8%


In [4]:
audit.multi_platform_player_days(df)

,DATE,PLAYER_ID,PLATFORM_ID,SESSIONS,SESSION_MINS,REVENUE,LIFETIME_REVENUE,PLATFORM_AGNOSTIC_LIFETIME_SPINS_DAYS
0,2026-09-22,224274,facebook,1.0,6.0832,NaN,0.0,1.0
1,2026-09-22,224274,sciwebgl,2.0,29.2255,NaN,0.0,1.0
2,2026-10-08,224274,facebook,1.0,4.7075,NaN,0.0,11.0
3,2026-10-08,224274,sciwebgl,1.0,3.7446,NaN,0.0,11.0
4,2026-10-09,224274,facebook,2.0,44.1891,NaN,0.0,12.0
5,2026-10-09,224274,sciwebgl,4.0,20.8184,NaN,0.0,12.0
6,2026-10-10,224274,facebook,2.0,4.0050,NaN,0.0,13.0
7,2026-10-10,224274,sciwebgl,2.0,2.2313,NaN,0.0,13.0
8,2026-10-11,224274,facebook,2.0,2.6752,NaN,0.0,14.0
9,2026-10-11,224274,sciwebgl,4.0,2.9659,NaN,0.0,14.0


**Finding: aggregated data. One row = player × platform × day on which the player was active.**

- `DATE + PLAYER_ID + PLATFORM_ID` is unique; no fully duplicated rows.
- 7 player-days (3 players) have two platform rows. Lifetime fields are identical across them; daily activity is split. **Decision:** roll up to one row per player per day (sum daily measures, take lifetime fields once).
- Almost every row has sessions and the login gap equals the gap between rows: **a missing row means the player did not play that day**, not missing data.
- A player appears once per active day (up to 116 rows), so model validation must group rows by player.

## 2. Coverage: dates, players, cohorts

In [5]:
daily = audit.daily_coverage(df)
print(f"{daily.index.min().date()} to {daily.index.max().date()}: {len(daily)} days, "
      f"{int(daily['missing_date'].sum())} without rows")
pc.daily_coverage(daily).show()

2026-05-09 to 2026-11-04: 180 days, 0 without rows


In [6]:
pc.active_by_cohort(audit.active_players_by_cohort(df)).show()

In [7]:
cohorts = audit.cohort_coverage(players, df['DATE'].max())
cohorts.style.format({'payer_share': '{:.0%}', 'first_install': '{:%Y-%m-%d}', 'last_install': '{:%Y-%m-%d}'})

,install_cohort,players,ever_payers,first_install,last_install,median_rows,max_possible_age_days,min_possible_age_days,payer_share
0,2026-05,100,6,2026-05-09,2026-05-31,2.000000,179,157,6%
1,2026-06,100,6,2026-06-01,2026-06-30,2.000000,156,127,6%
2,2026-07,100,4,2026-07-01,2026-07-31,2.000000,126,96,4%
3,2026-08,100,2,2026-08-01,2026-08-31,2.000000,95,65,2%
4,2026-09,100,3,2026-09-01,2026-09-30,1.000000,64,35,3%
5,2026-10,100,1,2026-10-01,2026-10-31,2.000000,34,4,1%
6,2026-11,100,1,2026-11-01,2026-11-04,1.000000,3,0,1%


In [8]:
pc.rows_per_player(audit.rows_per_player_distribution(players)).show()

In [9]:
activity = audit.cohort_activity_matrix(df, df['DATE'].max())
pc.cohort_activity_heatmap(activity, audit.AGE_LABELS).show()

**Finding: a stratified install-cohort sample.**

- 180 continuous days, 9 May – 4 Nov 2026. **Exactly 100 new players per month**, and 699 of 700 are first seen on install day.
- November's 100 installs all fall in Nov 1–4 (26–32 a day against 2–8), so the last days are distorted. Daily actives grow only because a new cohort joins every month.
- History is thin: 332 players (47%) were seen on a single day; the median is 2 rows.
- Oldest player: 169 days. **Nobody reaches day 365.** Only 23 players (3%) ever paid.
- About 30% of each cohort returns on day 1 and 13–16% are active at days 91–120; cohorts look alike.

## 3. Field groups and missing values

409 columns grouped by name (`src/sciplay/field_groups.py`). Hover a bar segment or a dot to see the columns.

In [10]:
pc.field_groups_bar(profile).show()

In [11]:
pc.missingness_strip(profile).show()

In [12]:
audit.missingness_by_group(profile).style.format({'median_null_share': '{:.0%}'})

,field_group,columns,median_null_share,all_null,constant,usable_columns
0,"Gameplay (spins, bets, features)",79,5%,6,13,60
1,"Game economy (coins, bankroll)",64,30%,8,3,53
2,"Device, app & performance",46,1%,2,3,41
3,Revenue & payers,42,97%,6,4,32
4,"Store, offers & purchase funnel",39,0%,0,6,33
5,Cohort milestones & outcomes,32,0%,0,5,27
6,Social & notifications,19,89%,4,2,13
7,Activity & engagement,17,0%,0,0,17
8,Errors & friction,16,1%,2,4,10
9,Dates & timestamps,16,36%,2,0,14


**Finding:** 85 of 409 columns carry no information (37 all null, 48 single value), including campaign and creative fields and the cheater and settlement flags. Most other nulls are **structural**, so three states are kept apart:

| Null in… | Means | Example |
|---|---|---|
| a daily count | **zero**: no event that day | `REVENUE` empty = no purchase |
| a last-event field | **not applicable**: never happened | `DAYS_SINCE_LAST_PURCHASE` empty = never paid |
| a dimension | **unknown** | `REGION` empty |

### Inconsistent semantics

*Semantics* is what a field actually means, as opposed to what its name says. A field has **inconsistent semantics** when the data contradicts its name or its siblings: a "365-day" field that is really spend so far, a "days" count that is only 0 or 1, or one yes/no concept stored three different ways. Each case below is tested on the data, because the brief warns that field names are clues, not definitions.

In [13]:
audit.inconsistent_semantics(df)

,field,name suggests,data shows,decision
0,D365_SPEND,Spend in the first 365 days,Equals LIFETIME_REVENUE on 100% of filled rows; oldest player is 169 days old,Excluded; not a valid target
1,D1_RETURNER … D90_RETURNER,Install milestone: came back by day n,"Played exactly n days after the row date: D1_RETURNER 100.0%, D7_RETURNER 99.9%, D30_RETURNER 100.0%",Excluded (future); possible churn label later
2,D30_SPEND,Spend in the first 30 days (fixed once day 30 has passed),Still changes after day 30 for 7 players,Excluded
3,D7 / D14 / D30_ACTIVE_DAYS,Number of active days,"Maximum value is 1: a yes/no flag, not a count",Excluded until verified
4,LIFETIME_REVENUE,Revenue up to and including the row date,Excludes the row date: today's REVENUE shows up in the next row 90% of the time,Kept; revenue to date = LIFETIME_REVENUE + REVENUE
5,LT_SPENDER_BUCKET,"Lifetime spender tier, consistent with LIFETIME_REVENUE","23 rows say 'Minnow' while LIFETIME_REVENUE = 0; 23 of them are first-purchase days (bucket is end of day, lifetime is start of day)",Reporting only
6,"IS_CHEATER, IS_CHEATER_DETECTED, CHEATS_DETECTED",One yes/no concept,Three encodings: IS_CHEATER = ['no']; IS_CHEATER_DETECTED = ['False']; CHEATS_DETECTED = ['0.0'],Normalised to one boolean in the eligibility filter
7,PLATFORM_ID = apple_sign_in,A platform,A login method (2 rows),Kept as a platform value (A5)


## 4. Column registry

Every column is placed in one of the six categories the brief asks to separate: **identifiers, dimensions, observed measures, derived features, targets, excluded**. Derived features and targets are not in the file; they are the columns we compute, so they are added as rows of their own.

Each column also gets an **as-of availability** (when its value is known relative to the row date D), a **null meaning** and a **use**. Rules are name-based (`src/sciplay/registry.py`), then corrected by evidence from the data. The full table is saved to `docs/column_registry.csv`.

In [14]:
reg_cat = registry.with_categories(reg)
registry.category_summary(reg_cat)

,category,columns,examples
0,Identifiers,5,"DATE, PLAYER_ID, PLATFORM_ID, GAME_ID, SLOT_ID"
1,Dimensions,35,"CHEATS_DETECTED, COUNTRY, DAU_TYPE, DEVICE_HARDWARE, DEVICE_OS_VERSION"
2,Observed measures,249,"ACTIVE_30D_PAYER_DAU, ACTIVE_90D_PAYER_DAU, BET_DENOM, BUY_COINS_CLICKED_BUYS, BUY_COINS_IMPRESSIONS"
3,Derived features,7,"revenue_to_date, age_days, days_since_last_seen, active_on_score_date, n_platforms"
4,Targets,2,"revenue_next_30d, ltv365"
5,Excluded,120,"ACCOUNTING_DATE, BUY_COINS_AWARD_IMPRESSIONS, BUY_COINS_PURCHASE_COMPLETES, BUY_COINS_PURCHASE_FAILED, CHURNERS"


In [15]:
reg_cat.loc[reg_cat['source'] == 'computed', ['column', 'category', 'as_of', 'use', 'note']]

,column,category,as_of,use,note
409,revenue_to_date,Derived features,end of day D,feature,LIFETIME_REVENUE (start of D) + REVENUE (day D)
410,age_days,Derived features,end of day D,feature,D - install date
411,days_since_last_seen,Derived features,end of day D,feature,D - date of the player's last row on or before D
412,active_on_score_date,Derived features,end of day D,feature,Has a row on D
413,n_platforms,Derived features,end of day D,feature,Platforms played on day D
414,MAIN_PLATFORM_ID,Derived features,end of day D,feature + segment / reporting,Platform with the most sessions on day D
415,install_cohort,Derived features,static (set at install),segment / reporting,Install month
416,revenue_next_30d,Targets,future (after D),training label,"Revenue from D+1 to D+30; no row = $0 (A1, A2). Only for D at least 30 days before the data end"
417,ltv365,Targets,future (after D),model output,Revenue to date + predicted revenue up to day 365 after install


In [16]:
reg_cat[reg_cat['category'] == 'Excluded'].groupby('use').size().rename('columns').reset_index()

,use,columns
0,exclude until verified,9
1,exclude: leakage,26
2,exclude: no information,80
3,exclude: not a valid target,2
4,exclude: redundant,1
5,freshness checks only,2


In [17]:
registry.usage_summary(reg)

use,Model feature,Derive feature,Segment / reporting,"Key, filter, metadata",Excluded: leakage / target,Excluded: no information,Total
field_group,,,,,,,
"Gameplay (spins, bets, features)",60,0,0,0,0,19,79
"Game economy (coins, bankroll)",53,0,0,0,0,11,64
"Device, app & performance",41,0,0,0,0,5,46
Revenue & payers,32,0,0,0,0,10,42
"Store, offers & purchase funnel",33,0,0,0,0,6,39
Cohort milestones & outcomes,0,0,0,0,27,5,32
Social & notifications,13,0,0,0,0,6,19
Activity & engagement,8,0,0,0,9,0,17
Errors & friction,10,0,0,0,0,6,16


In [18]:
reg_cat.to_csv(REPO_ROOT / 'docs' / 'column_registry.csv', index=False)
with pd.option_context('display.max_rows', 500):
    display(reg_cat.drop(columns=['rule']).style.format({'null_share': '{:.0%}'}, na_rep='–'))

,column,category,source,field_group,role,as_of,null_meaning,use,model_feature,null_share,n_unique,evidence,note
0,DATE,Identifiers,file,Dates & timestamps,timestamp,end of day D,n/a,key,False,0%,180.000000,,Row date D
1,PLAYER_ID,Identifiers,file,Keys & identifiers,identifier,static (set at install),n/a,key,False,0%,700.000000,,
2,PLATFORM_ID,Identifiers,file,Keys & identifiers,identifier,static (set at install),n/a,key,False,0%,6.000000,,
3,GAME_ID,Identifiers,file,Keys & identifiers,identifier,static (set at install),n/a,key,False,0%,1.000000,,
4,ACCOUNTING_DATE,Excluded,file,Dates & timestamps,timestamp,end of day D,n/a,exclude: redundant,False,0%,7.000000,,Month of D
5,ACTIVE_30D_PAYER_DAU,Observed measures,file,Revenue & payers,daily measure,end of day D,zero: no event that day,feature,True,0%,2.000000,,
6,ACTIVE_90D_PAYER_DAU,Observed measures,file,Revenue & payers,daily measure,end of day D,zero: no event that day,feature,True,0%,2.000000,,
7,BET_DENOM,Observed measures,file,"Gameplay (spins, bets, features)",daily measure,end of day D,zero: no event that day,feature,True,13%,21.000000,,
8,BUY_COINS_AWARD_IMPRESSIONS,Excluded,file,"Store, offers & purchase funnel",daily measure,end of day D,zero: no event that day,exclude: no information,False,100%,1.000000,,Single value
9,BUY_COINS_CLICKED_BUYS,Observed measures,file,"Store, offers & purchase funnel",daily measure,end of day D,zero: no event that day,feature,True,97%,8.000000,,


## 5. As-of policy

**A score for day D (as of the end of D, UTC) uses only rows dated D or earlier, and only columns the registry marks as features.** The future may appear in the label, never in a feature.

**Revenue to date** at the end of D = `LIFETIME_REVENUE` (start of D) + `REVENUE` (day D).

The policy is code (`src/sciplay/asof.py`): `rows_as_of`, `snapshot_as_of`, `feature_columns` and `assert_as_of`, which fails if a feature frame holds a row after D or a column the registry excludes.

### Example A · an active player

Player 289987 installed on 1 Oct. We score him for **D = 8 Oct**.

In [19]:
asof.player_timeline(df, 289987, '2026-10-08',
                     ['DAYS_SINCE_INSTALL', 'SESSIONS', 'REVENUE', 'LIFETIME_REVENUE', 'D1_RETURNER', 'NEXT_LOGIN_DATE'])

,DATE,row_usable_for_score,DAYS_SINCE_INSTALL,SESSIONS,REVENUE,LIFETIME_REVENUE,D1_RETURNER,NEXT_LOGIN_DATE,revenue_to_date_end_of_day
0,2026-10-01,yes,0,3.0,45.28,0.00,1,2026-10-02,45.28
1,2026-10-02,yes,1,2.0,NaN,45.28,1,2026-10-03,45.28
2,2026-10-03,yes,2,3.0,5.52,45.28,1,2026-10-04,50.80
3,2026-10-04,yes,3,2.0,NaN,50.80,1,2026-10-05,50.80
4,2026-10-05,yes,4,2.0,NaN,50.80,1,2026-10-06,50.80
5,2026-10-06,yes,5,1.0,NaN,50.80,1,2026-10-07,50.80
6,2026-10-07,yes,6,1.0,NaN,50.80,1,2026-10-08,50.80
7,2026-10-08,yes,7,6.0,8.84,50.80,1,2026-10-09,59.64
8,2026-10-09,no: after D,8,9.0,17.67,59.64,1,2026-10-10,77.31
9,2026-10-10,no: after D,9,1.0,NaN,77.31,1,2026-10-11,77.31


- **Used:** rows 1–8 Oct, including 8 Oct itself: 6 sessions and $8.84 spent that day.
- **Used:** revenue to date = $50.80 (lifetime at the start of 8 Oct) + $8.84 = **$59.64**. The 9 Oct row confirms it: its start-of-day lifetime revenue is exactly $59.64.
- **Not used:** the 9 Oct row and its $17.67 purchase. That is the future.
- **Not used:** `D1_RETURNER = 1` and `NEXT_LOGIN_DATE = 9 Oct` on the 8 Oct row. Both describe 9 Oct, which nobody knows on the night of 8 Oct (section 6).
- **Label (training only):** his revenue from 9 Oct to 7 Nov. The future is allowed here.

**Missing target ≠ zero by default.** The brief warns against filling a missing target with 0. Here it is supported: a day without a row is a day without play (A1), and an empty `REVENUE` is a day without a purchase (A2, verified), so revenue in the label window is a true $0 for a player who did not pay. The label is only built for score dates whose 30-day window ends inside the data; later dates have *no* label, not a zero one. `D365_SPEND` being empty for non-payers is not used at all.

### Example B · an inactive player

Player 160253's last row is 6 Sep. We score everyone for **D = 10 Sep**.

In [20]:
player_day = asof.rollup_player_day(df, reg)
snap = asof.snapshot_as_of(player_day, '2026-09-10')
cols = ['score_date', 'PLAYER_ID', 'last_seen', 'days_since_last_seen', 'active_on_score_date',
        'revenue_to_date', 'age_days']
snap.loc[snap['PLAYER_ID'] == 160253, cols]

,score_date,PLAYER_ID,last_seen,days_since_last_seen,active_on_score_date,revenue_to_date,age_days
378,2026-09-10,160253,2026-09-06,4,False,84.48,13


In [21]:
print(f"Players known by 10 Sep: {len(snap)}; with a row on 10 Sep: {int(snap['active_on_score_date'].sum())}")

Players known by 10 Sep: 440; with a row on 10 Sep: 30


He has no row on 10 Sep, so the as-of snapshot carries his 6 Sep row forward: $84.48 revenue to date, 4 days since last seen. Without this, only the 30 players who played that day would be scored, and the players drifting away, the ones the lists are for, would disappear.

### The policy as a check

In [22]:
allowed = asof.feature_columns(reg)
ok = player_day.loc[player_day['DATE'] <= '2026-10-08', ['DATE', 'PLAYER_ID'] + allowed[:5]]
asof.assert_as_of(ok, '2026-10-08', allowed)
print(f'{len(allowed)} feature columns allowed; frame passes')

try:
    bad = player_day.loc[player_day['DATE'] <= '2026-10-08', ['DATE', 'PLAYER_ID', 'D1_RETURNER']]
    asof.assert_as_of(bad, '2026-10-08', allowed)
except ValueError as err:
    print('Rejected:', err)

259 feature columns allowed; frame passes
Rejected: columns not allowed by the registry: ['D1_RETURNER']


## 6. Leakage audit

A column leaks when its value on the row for day D depends on what happens after D. The registry marks each suspicious column and the decision; the evidence is below.

### Example: what `D1_RETURNER` really means

The same player has different `D1_RETURNER` values on different rows, so it is not an install milestone. It is 1 exactly when the player has a row on the next day:

In [23]:
asof.returner_flag_example(df, 39593)

,DATE,D1_RETURNER,NEXT_LOGIN_DATE,next_row_date,played_next_day
0,2026-05-12,0,2026-05-16,2026-05-16,0
1,2026-05-16,1,2026-05-17,2026-05-17,1
2,2026-05-17,0,2026-05-19,2026-05-19,0
3,2026-05-19,0,2026-05-26,2026-05-26,0
4,2026-05-26,1,2026-05-27,2026-05-27,1
5,2026-05-27,0,2026-06-02,2026-06-02,0


Tested on every row and every horizon, `D<n>_RETURNER` = *the player played exactly n days after D*:

In [24]:
registry.returner_label_evidence(df).style.format({'match_share': '{:.1%}'})

,column,days_after_D,rows_tested,match_share
0,D1_RETURNER,1,4176,100.0%
1,D3_RETURNER,3,4041,100.0%
2,D7_RETURNER,7,3893,99.9%
3,D14_RETURNER,14,3656,99.9%
4,D21_RETURNER,21,3433,100.0%
5,D30_RETURNER,30,3114,100.0%
6,D60_RETURNER,60,2167,100.0%
7,D90_RETURNER,90,1362,100.0%


In [25]:
registry.milestone_backfill_evidence(df).query('positive_before_age > 0')

,column,age_days,rows_younger_than_age,positive_before_age
1,D1,1,702,213
2,D1_RETURNER,1,702,213
3,D14,14,1795,54
4,D14_CONVERTER,14,1795,76
5,D14_RETURNER,14,1795,394
6,D21,21,2121,43
7,D21_RETURNER,21,2121,445
8,D3,3,1058,91
9,D3_CONVERTER,3,1058,19
10,D3_RETURNER,3,1058,196


### Decisions

In [26]:
suspicious = reg[reg['use'].str.startswith('exclude: leak') | reg['use'].isin(
    ['exclude: not a valid target', 'exclude until verified'])
    | reg['column'].isin(['LIFETIME_REVENUE', 'LAST_4W_REVENUE', 'LAST_8W_REVENUE', 'LAST_12W_REVENUE',
                          'DAYS_SINCE_LAST_LOGIN', 'DAYS_SINCE_LAST_PURCHASE', 'LT_PAYER_BIN',
                          'TOTAL_SPEND_LAST_4W_SEGMENT'])]
with pd.option_context('display.max_rows', 100):
    display(suspicious[['column', 'as_of', 'use', 'evidence', 'note']].reset_index(drop=True))

,column,as_of,use,evidence,note
0,D0_CONVERTER,back-filled after D,exclude: leakage,,"Forward-looking label: D<n>_RETURNER = played exactly n days after D (Q3, Q5)"
1,D1,back-filled after D,exclude: leakage,positive on 213 rows before day 1,"Forward-looking label: D<n>_RETURNER = played exactly n days after D (Q3, Q5)"
2,D1_RETURNER,back-filled after D,exclude: leakage,positive on 213 rows before day 1,"Forward-looking label: D<n>_RETURNER = played exactly n days after D (Q3, Q5)"
3,D14,back-filled after D,exclude: leakage,positive on 54 rows before day 14,"Forward-looking label: D<n>_RETURNER = played exactly n days after D (Q3, Q5)"
4,D14_CONVERTER,back-filled after D,exclude: leakage,positive on 76 rows before day 14,"Forward-looking label: D<n>_RETURNER = played exactly n days after D (Q3, Q5)"
5,D14_RETURNER,back-filled after D,exclude: leakage,positive on 394 rows before day 14,"Forward-looking label: D<n>_RETURNER = played exactly n days after D (Q3, Q5)"
6,D21,back-filled after D,exclude: leakage,positive on 43 rows before day 21,"Forward-looking label: D<n>_RETURNER = played exactly n days after D (Q3, Q5)"
7,D21_RETURNER,back-filled after D,exclude: leakage,positive on 445 rows before day 21,"Forward-looking label: D<n>_RETURNER = played exactly n days after D (Q3, Q5)"
8,D3,back-filled after D,exclude: leakage,positive on 91 rows before day 3,"Forward-looking label: D<n>_RETURNER = played exactly n days after D (Q3, Q5)"
9,D3_CONVERTER,back-filled after D,exclude: leakage,positive on 19 rows before day 3,"Forward-looking label: D<n>_RETURNER = played exactly n days after D (Q3, Q5)"


| Columns | Why suspicious | Decision |
|---|---|---|
| `D1`–`D90_RETURNER`, `D<n>`, `D<n>_CONVERTER`, `SPENDER_<n>` | Describe activity n days after D | **Excluded** (usable later as churn labels) |
| `NEXT_LOGIN_DATE`, `DAYS_UNTIL_NEXT_PURCHASE`, `CHURNERS*` | Describe what happens after D | **Excluded** |
| `D365_SPEND`, `D30_SPEND` | Lifetime spend so far, not a fixed window | **Excluded**, not a valid target |
| `D7/D14/D30_ACTIVE/PURCHASE/SPINS_DAYS` | Window may run past D | **Excluded until verified** |
| `LIFETIME_REVENUE` | Start-of-day value | **Kept**, plus same-day `REVENUE` |
| `LAST_4W/8W/12W_*`, `DAYS_SINCE_*` | Rolling; window end not documented | **Kept**, used as given |
| Spend segments (`LT_PAYER_BIN`, `TOTAL_SPEND_LAST_4W_SEGMENT`) | Built from spend | **Reporting only** |

## 7. Late data, multiple platforms, sparse fields

In [27]:
lag = audit.late_updates(df)
pc.update_lag(audit.update_lag_rows(df)).show()
lag.style.format({'share_over_7d': '{:.0%}'})

,data_month,rows,median_lag,max_lag,share_over_7d
0,2026-05,276,104.000000,171,100%
1,2026-06,409,91.000000,154,100%
2,2026-07,540,91.000000,125,100%
3,2026-08,804,80.000000,94,100%
4,2026-09,895,50.000000,65,100%
5,2026-10,1067,21.000000,35,91%
6,2026-11,254,2.000000,4,0%


In [28]:
span_days = (players['last_seen'] - players['install_date']).dt.days + 1
pd.Series({
    'Player-days with two platform rows': int(df.duplicated(['DATE', 'PLAYER_ID']).sum()),
    'Days played / days between install and last seen': round(players['active_days'].sum() / span_days.sum(), 3),
    'Rows with an empty REVENUE (no purchase)': round(df['REVENUE'].isna().mean(), 3),
}, name='value').to_frame()

,value
Player-days with two platform rows,7.000
Days played / days between install and last seen,0.360
Rows with an empty REVENUE (no purchase),0.971


| | What we see | Effect on calculations | Handling |
|---|---|---|---|
| **Late-arriving data** | Recent rows land ~2 days late; every row changes until 91 days after its date | Yesterday may be missing; forward-looking flags on recent rows keep changing | Score D only once its rows landed (freshness check); features only from as-of-safe columns; labels only after their window has passed |
| **Multiple platforms** | 7 player-days with two rows | Counting rows double counts players and splits a day's play | One row per player per day; main platform + platform count |
| **Sparse event fields** | No row on days off; `REVENUE` empty on 97% of rows | Days off look like missing data; averages over rows overstate activity; zeros dominate revenue | As-of snapshot carries players forward; empty daily count = 0, empty last-event = not applicable; two-part revenue model |

## 8. Data quality issues

Six issues, grouped into four themes. Each could change a business decision if ignored.

In [29]:
audit.quality_issues(df)

,id,theme,issue,evidence,business_impact
0,Q1,Target validity,"D365_SPEND is lifetime-to-date spend, not a 365-day outcome",Equals LIFETIME_REVENUE on 100% of filled rows; null exactly when the player never paid; changes across snapshots for 12 of 21 payers; oldest row is day 169 after install,Training on it would teach the model that LTV365 equals spend so far: systematic underprediction for young players and wrong UA / CRM budgets
1,Q2,Revenue definition,REVENUE and LIFETIME_REVENUE are offset by one day,Today's REVENUE matches the change into the same row 9% of the time and the change into the next row 90% of the time,Adding REVENUE to LIFETIME_REVENUE double counts; using only LIFETIME_REVENUE misses today's purchases. LTV-to-date and the opportunity gap shift by a day
2,Q3,Time integrity (as-of policy),Rows keep being rewritten for 91 days after their data date,"Median days from data date to LAST_UPDATE_DATETIME, by data month: 2026-05: 104d, 2026-06: 91d, 2026-07: 91d, 2026-08: 80d, 2026-09: 50d, 2026-10: 21d, 2026-11: 2d","Each row is updated until 91 days after its date, then freezes: D90_RETURNER can only be filled on D+90. Same mechanism as Q5. Once forward-looking columns are excluded the training data is safe; ..."
3,Q4,Sample limitations,Stratified sample: a fixed number of installs per month,"Players per install cohort: 2026-05: 100, 2026-06: 100, 2026-07: 100, 2026-08: 100, 2026-09: 100, 2026-10: 100, 2026-11: 100","Totals and pay rates are not population figures; dashboard totals must be labelled as sample totals, and the newest cohort is only days old"
4,Q5,Time integrity (as-of policy),Forward-looking and back-filled fields sit next to features,"D<n>_RETURNER means 'played exactly n days after D' (100% match), so D1_RETURNER is already 1 on 213 day-0 rows; NEXT_LOGIN_DATE is after DATE on 100% of filled rows; D30_SPEND changes after day 3...","Leaky features inflate offline accuracy and fail silently in production, so lists would flag the wrong players"
5,Q6,Sample limitations,Cheater and legal-settlement flags are never set,"IS_CHEATER, IS_CHEATER_DETECTED and IS_LEGAL_SETTLEMENT have a single value ('no' / False)",The exclusion policy cannot be tested on this sample; in production an untested filter could push offers to excluded players


In [30]:
pc.d365_vs_lifetime(df).show()

In [31]:
pc.revenue_offset(df).show()

| Theme | Issues | Resolved by |
|---|---|---|
| Target validity | Q1 | Defining LTV365 ourselves (observed + modelled remaining revenue); `D365_SPEND` is never a label |
| Revenue definition | Q2 | Revenue to date = `LIFETIME_REVENUE` (start of D) + `REVENUE` (day D) |
| Time integrity | Q3, Q5 | The as-of policy and registry (sections 5–6). Training and testing are safe; auditing past lists relies on the stored daily list history |
| Sample limitations | Q4, Q6 | Rates instead of totals; uncertainty shown on LTV; the eligibility filter is built and unit-tested on synthetic flags |

## 9. Definitions

In [32]:
components = ['BUY_COINS_REVENUE', 'HOT_OFFER_REVENUE', 'OOC_REVENUE']
check = (df[components].fillna(0).sum(axis=1) - df['REVENUE'].fillna(0)).abs().lt(0.01).mean()
print(f'REVENUE equals coin packs + hot offers + out-of-coins offers on {check:.0%} of rows')
print(f"Net revenue exists only for direct-to-consumer purchases: "
      f"${df['DTC_NET_FEE_REVENUE'].sum():,.2f} net of ${df['DTC_REVENUE'].sum():,.2f} gross")

REVENUE equals coin packs + hot offers + out-of-coins offers on 100% of rows
Net revenue exists only for direct-to-consumer purchases: $19.34 net of $20.80 gross


| Term | Unit | Business meaning | Calculation |
|---|---|---|---|
| **Revenue** | Gross USD per player per day | In-app purchases that day, before store fees and refunds; no ad revenue in the data | `REVENUE` = coin packs + hot offers + out-of-coins offers; empty = $0 |
| **Revenue to date** | USD per player | Everything paid since install, up to the end of D | `LIFETIME_REVENUE` (start of D) + `REVENUE` (day D) |
| **LTV365** | USD per player | Expected gross revenue in the first year after install | Revenue to date + modelled revenue up to day 365; cohort LTV = average |
| **Active player** | Player on a day | Played that day | Has a row with sessions on D; DAU = count of active players |
| **Payer** | Player | Ever-payer: paid at least once. Recent payer: paid in the last 4 weeks | Revenue to date > 0; `LAST_4W_REVENUE` > 0 |
| **Underperformance** | Player on score date D | Clearly less play or spend than expected, for a reason a team can act on | One list per player, in this order, ranked by value at risk |

**The three lists**

1. **Friction:** technical problems in the last 7 days: failed purchase or store verification, crash, unhandled error, support ticket. Cancelled purchases are *not* friction (mostly players browsing the store: 114 players, few of them payers); they appear as a supporting reason on the other lists.
2. **Revenue decline:** a payer whose last 4 weeks' spend is at least 50% below the 4 weeks before (`LAST_4W_REVENUE` vs `LAST_8W_REVENUE` − `LAST_4W_REVENUE`), or who is well past their usual purchase gap.
3. **Engagement decline:** plays well below their own 28-day pace (`PLATFORM_AGNOSTIC_LAST_7_DAYS_PLAYED` vs `..._LAST_28_...` ÷ 4) or the median player of the same age, or is well past their usual gap.

**Eligibility:** not a cheater or legal settlement, at least 3 days since install, seen in the last 30 days. **Value at risk** = predicted LTV365 − revenue to date (USD). Thresholds live in a config file and are tuned on list sizes.

## 10. Summary

| Topic | Finding | What it means for the next steps |
|---|---|---|
| Grain | Player × platform × active day, aggregated | Roll up to player-day; as-of snapshot for scoring |
| Coverage | 180 days; 700 players, 100 new per month; 23 payers; max age 169 days | No mature 365-day outcome; few payers; sample totals |
| Columns | Six categories; 259 candidate features, 37 excluded for leakage or target, 81 with no information | Registry drives feature selection |
| Semantics | 8 fields whose meaning differs from their name | Tested, not assumed; decisions recorded |
| As-of policy | Rows ≤ D, columns known by end of D | Enforced in code with `assert_as_of` |
| Leakage | D-returner flags = activity n days after D | Excluded; usable later as churn labels |
| Quality | Four themes | LTV365 is a defined proxy; time integrity by policy; sample limits stated |